## 1장 2강 : LLM의 한계와 위험 요소

### 2. LLM의 4대 한계와 발생 원인

### 2.1 주요 한계 개념 정리
- 환각 (Hallucination)
  - LLM은 분류 모델(토큰에 대한 분류 모델)
- 지식 컷오프 (Knowledge Cutoff)
- 비결정성 (Non-determinism)
  - 같은 질문을 해도 다른 답변이 나올 수 있음
- 편향 (Bias)

#### 2.2 파이썬 실습: 환각 방지 프롬프트 및 temperature 제어

- top-k, top-p, temperature
- top-k: 가장 높은 확률의 k개 만큼의 토큰을 선택, 선택된 토큰을 응답으로 사용 Ex) top-k= 100 -> 가장 높은 확률 100개 중에서 선택
- top-p: p확률 이상의 토큰에서 선택
- top-p + top-k
- temperature: 확률 분포를 조정 (0에 가까울수록 정확한 답변, 1에 가까울수록 창의적인 답변)
  - 0에 가깝게 설정: 도구선택, 형식의 고정 

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
    ("system", "당신은 객관적 사실만을 전달하는 정보 검증관입니다. 제공된 지식 범위 밖의 내용이거나 사실 확인이 불가능하다면 '해당 정보는 확인할 수 없습니다'라고만 명확히 답변하세요. 절대 거짓 정보를 지어내지 마세요."),
    ("user", "{user_question}")
])


model = init_chat_model(
    model_provider="ollama",
    model="mistral",
    temperature=0
)

parser = StrOutputParser()

chain = prompt_template | model | parser

question = "대한민국의 수도는 서울이다"
response = chain.invoke({"user_question": question})
response

' 해당 정보는 확인할 수 있습니다. 대한민국의 수도는 서울입니다.'

In [ ]:
# 순수 텍스트 추출을 위한 출력 파서를 선언합니다.


In [ ]:
# LECL 파이프(|) 연산자로 실행 체인을 결합합니다.


In [ ]:
# 실제로 존재하지 않는 허구의 사건을 질문으로 전달합니다.


# 체인을 호출하고 환각 억제 결과를 확인합니다.


### 3. 민감 정보 유출 위험과 데이터 전처리

#### 3.1 민감 정보 입력의 위험성
- 외부 유출 가능성
- 검증 필요 업무

#### 3.2 정규식 기반 개인정보 마스킹 함수

In [12]:
import re

# 정규 표현식을 활용하여 전화번호와 이메일을 익명화하는 함수를 정의합니다.
def mask_personal_information(text: str) -> str:
    # 전화번호 형식 (예: 010-1234-5678)을 감지하는 패턴입니다.
    phone_pattern = re.compile(r'\d{3}-\d{4}-\d{4}')

    # 이메일 형식 (예: user@example.com)을 감지하는 패턴입니다.
    email_pattern = re.compile(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+')

    # 일치하는 패턴을 대체 문구로 치환합니다.
    masked = phone_pattern.sub("[전화번호 보호]", text)
    masked = email_pattern.sub("[이메일 보호]", masked)
    return masked

In [13]:
user_input = "연락처는 010-1111-2222, 이메일은 dlapdlf@email.com"
mask_personal_information(user_input)

'연락처는 [전화번호 보호], 이메일은 [이메일 보호]'